# PCS Workshop Intro — Build a Neural Network That Reads Handwriting  ✍️🔢

You are about to build a neural network in **one Python file** and teach it to
recognise handwritten digits. You write every line that matters — the layers, the
loss, the update — and PyTorch only keeps score.

Keep this loop in mind the whole time:

> **Forward pass → Loss → Backpropagation → Update weights → Repeat**

**What you do:**

1. Run the setup cell below once.
2. Open `mnist_network.py` in the file panel and build it with the slides.
3. Save, then run each checkpoint cell here to prove that stage works.
4. At the end, run it for real and watch a digit get read.
5. Download `mnist_network.py` — that file is your work.

> **Do not use File → Save.** Your work is the Python file, not this notebook. Use the download cell at the bottom.

## 1. Set up your workbench  ·  run once

Leave the runtime on the standard **CPU** setting — no GPU needed. This cell
downloads the workshop, creates your editable file, and pre-loads MNIST so the
data is ready when your code needs it.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

import torch
import torchvision

REPO_URL = "https://github.com/uh-pcs/Workshop3.git"
REPO_DIR = Path("/content/Workshop3")
WORK_FILE = Path("/content/mnist_network.py")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if not WORK_FILE.exists():
    shutil.copy(REPO_DIR / "starter/mnist_network.py", WORK_FILE)
    file_status = "created your starter file"
else:
    file_status = "kept your existing file (setup is safe to re-run)"

# optional helpers for the payoff cells, refreshed every run
shutil.copy(REPO_DIR / "solution/inspect_model.py", "/content/inspect_model.py")

try:
    for train in (True, False):
        torchvision.datasets.MNIST("/content/data", train=train, download=True)
    data_status = "MNIST ready"
except Exception as error:
    data_status = f"MNIST DOWNLOAD FAILED — {error!r}\n   Re-run this cell, or switch networks."

print(f"Python {sys.version.split()[0]}  ·  PyTorch {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}  (we use CPU on purpose)")
print(f"Workbench: {file_status}")
print(data_status)
print()
print("READY — open mnist_network.py from the file panel on the left." if data_status == "MNIST ready"
      else "SETUP NEEDS ATTENTION — see the line above.")

## 2. Open your Python file

In the **file panel** on the left (the folder icon), double-click
**`mnist_network.py`**. Keep this notebook open beside it — you will build the
file with the slides and come back here at each checkpoint.

## Checkpoint 1 — your `Linear` layer

Built the `Linear` class? **Save the file**, then run this. It imports your file
in a brand-new Python process, so it always tests exactly what you saved.

In [ ]:
# CHECKPOINT 1 — three images through one Linear layer
check = r"""
import torch
try:
    from mnist_network import Linear
except Exception as e:
    raise SystemExit(f"Can't import Linear yet - is the class written and the file saved?\n   {e!r}")
out = Linear(784, 128).forward(torch.zeros(3, 784))
if tuple(out.shape) != (3, 128):
    raise SystemExit(f"Output shape is {tuple(out.shape)}, expected (3, 128).\n"
                     "   The weight matrix is (in_features, out_features) = (784, 128).")
print("Linear:  (3, 784) -> (3, 128)")
"""
import subprocess, sys
done = subprocess.run([sys.executable, "-c", check], cwd="/content").returncode == 0
print("\n" + ("Checkpoint 1 passed - save and keep going."
              if done else "Not passing yet. Read the note above, fix the file, save, run again."))

## Checkpoint 2 — your whole network

Stacked the three `Linear` layers and two `ReLU`s into `NeuralNetwork`? Save,
then check that four images come out as ten digit scores each.

In [ ]:
# CHECKPOINT 2 — four images become ten scores each
check = r"""
import torch
try:
    from mnist_network import NeuralNetwork
except Exception as e:
    raise SystemExit(f"Can't import NeuralNetwork yet - written and saved?\n   {e!r}")
model = NeuralNetwork(784, 128, 10)
out = model.forward(torch.zeros(4, 784))
if tuple(out.shape) != (4, 10):
    raise SystemExit(f"Output shape is {tuple(out.shape)}, expected (4, 10). "
                     "Trace 784 -> 128 -> 128 -> 10.")
n = len(model.parameters())
if n != 6:
    raise SystemExit(f"parameters() returned {n} tensors, expected 6 "
                     "(three weights + three biases). Are you returning a flat list?")
print("Network:  (4, 784) -> (4, 10)   with 6 trainable tensors")
"""
import subprocess, sys
done = subprocess.run([sys.executable, "-c", check], cwd="/content").returncode == 0
print("\n" + ("Checkpoint 2 passed - save and keep going."
              if done else "Not passing yet. Read the note above, fix the file, save, run again."))

## Checkpoint 3 — one learning step

This runs the full loop once on a tiny fixed problem — forward, loss, backward,
update, clear — and checks the loss actually went **down**. If it does, your
learning loop works and the real thing will too.

In [ ]:
# CHECKPOINT 3 — one manual step must reduce the loss
check = r"""
import torch
try:
    from mnist_network import NeuralNetwork
except Exception as e:
    raise SystemExit(f"Can't import NeuralNetwork - written and saved?\n   {e!r}")
torch.manual_seed(0)
model = NeuralNetwork(2, 4, 2)
x = torch.tensor([[2.0, 0.0], [0.0, 2.0]])
target = torch.tensor([0, 1])
loss_before = torch.nn.functional.cross_entropy(model.forward(x), target)
loss_before.backward()
for p in model.parameters():
    if p.grad is None:
        raise SystemExit("A parameter has no gradient. Did loss.backward() run, "
                         "and do weights/biases call requires_grad_()?")
with torch.no_grad():
    for p in model.parameters():
        p -= 0.1 * p.grad
for p in model.parameters():
    p.grad.zero_()
loss_after = torch.nn.functional.cross_entropy(model.forward(x), target)
if not (loss_after < loss_before):
    raise SystemExit(f"Loss went {loss_before.item():.4f} -> {loss_after.item():.4f} (not down). "
                     "Check the update sign (-=) and that grads are cleared.")
print(f"Learning:  loss {loss_before.item():.4f} -> {loss_after.item():.4f}")
"""
import subprocess, sys
done = subprocess.run([sys.executable, "-c", check], cwd="/content").returncode == 0
print("\n" + ("Checkpoint 3 passed - your learning loop works."
              if done else "Not passing yet. Read the note above, fix the file, save, run again."))

## 3. Run it for real — watch it learn

Finished `main()`? Save, then run this. It trains for 10 epochs and prints the
loss and accuracy as it goes. Blind guessing on ten digits is ~10% — anything
climbing well past that is your network actually learning.

Then change `epochs = 10` to `epochs = 60` in the file, save, and run this again.

In [ ]:
# Final run: execute exactly the file you built, in a fresh process.
import subprocess, sys
subprocess.run([sys.executable, "/content/mnist_network.py"], cwd="/content", check=True)

## 4. See it read one digit  🔎

The run above happened in a separate process, so nothing survived. This trains
the same network again *here* in the notebook (~1 minute) so we can look inside
it — then draws one test image as text with the model's guess.

In [ ]:
import importlib
import inspect_model
importlib.reload(inspect_model)

model, x_test, target_test, history = inspect_model.quick_train(epochs=15)

In [ ]:
i = 0  # change this to see other test images
guess = inspect_model.predict(model, x_test[i:i + 1])[0].item()
inspect_model.show_digit(x_test[i], guess=guess, truth=target_test[i].item())

## 5. The learning curve

Loss goes down, accuracy goes up — past the 10% blind-guess line.

In [ ]:
inspect_model.plot_history(history)

## 6. Where it gets things wrong

Your network matches pixel patterns — it has no idea what a digit *means*. These
are the test images it got wrong while feeling **most confident** (often a 4 that
looks like a 9, or a 5 that looks like a 3).

In [ ]:
for i in inspect_model.worst_mistakes(model, x_test, target_test, k=3):
    guess = inspect_model.predict(model, x_test[i:i + 1])[0].item()
    inspect_model.show_digit(x_test[i], guess=guess, truth=target_test[i].item())
    print("-" * 28)

## Done!  🎉

You built a neural network from tensors up — the learned matrix multiplication,
the activation, the loss, the gradients, and the update — and taught it to read
handwriting. No `torch.nn.Linear`, no optimizer, nothing hidden.

> **The one idea:** it made a guess, measured how wrong it was, changed its
> weights a little, and repeated.

**Keep experimenting:** [`EXTENSIONS.md`](https://github.com/uh-pcs/Workshop3/blob/main/EXTENSIONS.md)
— change the epochs, the learning rate, the hidden size; break the initialisation;
or look at the pictures the first layer learned with
`inspect_model.layer1_weight_grid(model)`.

## 7. Keep your file

The Colab runtime is temporary, and **File → Save does not save your `.py` file** (it would only try to write this notebook back to GitHub). Download `mnist_network.py` now.

In [ ]:
from google.colab import files
files.download("/content/mnist_network.py")

---

## Catch up  ·  only when the facilitator says so

If you fall behind, these restore a known-good `mnist_network.py` so you rejoin
the room for the payoff. `recover(1..3)` matches the three checkpoints;
`recover(4)` is the complete file.

**This replaces your current file** — if you want to keep your attempt, run the
download cell above first.

In [ ]:
def recover(stage):
    names = {1: "01-linear.py", 2: "02-network.py",
             3: "03-learning-step.py", 4: "04-complete.py"}
    if stage not in names:
        raise ValueError("stage must be 1, 2, 3, or 4")
    shutil.copy(REPO_DIR / "facilitator/checkpoints" / names[stage], WORK_FILE)
    print(f"Recovered stage {stage}  ({names[stage]}) - reopen the file to see it.")

In [ ]:
recover(1)   # after the Linear checkpoint

In [ ]:
recover(2)   # after the whole network

In [ ]:
recover(3)   # after the learning step

In [ ]:
recover(4)   # the complete workshop solution